In [1]:
import pandas as pd
import xgboost as xgb

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    roc_auc_score
)

In [3]:
# Load and read the data
df = pd.read_csv("train_transaction.csv")

print("Dataset shape:", df.shape)
print(df.head())

Dataset shape: (64437, 394)
   TransactionID  isFraud  TransactionDT  TransactionAmt ProductCD  card1  \
0        2987000        0          86400            68.5         W  13926   
1        2987001        0          86401            29.0         W   2755   
2        2987002        0          86469            59.0         W   4663   
3        2987003        0          86499            50.0         W  18132   
4        2987004        0          86506            50.0         H   4497   

   card2  card3       card4  card5  ... V330  V331  V332  V333  V334 V335  \
0    NaN  150.0    discover  142.0  ...  NaN   NaN   NaN   NaN   NaN  NaN   
1  404.0  150.0  mastercard  102.0  ...  NaN   NaN   NaN   NaN   NaN  NaN   
2  490.0  150.0        visa  166.0  ...  NaN   NaN   NaN   NaN   NaN  NaN   
3  567.0  150.0  mastercard  117.0  ...  NaN   NaN   NaN   NaN   NaN  NaN   
4  514.0  150.0  mastercard  102.0  ...  0.0   0.0   0.0   0.0   0.0  0.0   

  V336  V337  V338  V339  
0  NaN   NaN   NaN 

In [16]:
#Select only numeric columns
numeric_columns = df.select_dtypes(include=["int64", "float64"]).columns

df = df[numeric_columns]

In [5]:
#target aur featues ko alag kar diya
X = df.drop("isFraud", axis=1)
y = df["isFraud"]


In [6]:
#missing value fill karde
X = X.fillna(X.median())

In [7]:
#data split train or test mai
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)


In [8]:
#imbalance check
normal = (y_train == 0).sum()
fraud = (y_train == 1).sum()

scale_pos_weight = normal / fraud

print("Normal transactions:", normal)
print("Fraud transactions:", fraud)
print("Scale Pos Weight:", scale_pos_weight)

Normal transactions: 50175
Fraud transactions: 1374
Scale Pos Weight: 36.5174672489083


In [9]:
#Creating XGBoost model
model = xgb.XGBClassifier(
    n_estimators=100,
    max_depth=5,
    learning_rate=0.1,
    scale_pos_weight=scale_pos_weight,
    eval_metric="logloss",
    random_state=42
)

In [10]:
#train the model
model.fit(X_train, y_train)

print("Model training completed!")

Model training completed!


In [11]:
y_pred = model.predict(X_test)

In [12]:
y_probability = model.predict_proba(X_test)[:, 1]

In [13]:
print("\nClassification Report:")
print(classification_report(y_test, y_pred))



Classification Report:
              precision    recall  f1-score   support

           0       0.99      0.92      0.95     12545
           1       0.20      0.75      0.31       343

    accuracy                           0.91     12888
   macro avg       0.59      0.83      0.63     12888
weighted avg       0.97      0.91      0.94     12888



In [14]:
print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred))


Confusion Matrix:
[[11491  1054]
 [   87   256]]


In [15]:
roc_auc = roc_auc_score(y_test, y_probability)

print("\nROC-AUC Score:", roc_auc)


ROC-AUC Score: 0.9109581018537346
